# 6-1절 연습 문제 풀이

이 노트북은 6-1절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch06/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 6장 연습에서 공통으로 사용하는 시 텍스트와 도구
POEM = '엄마야 누나야 강변 살자 뜰에는 반짝이는 금모래빛 뒷문밖에는 갈잎의 노래 엄마야 누나야 강변 살자'
EOS = '<eos>'

def build_vocab(tokens):
    vocab = {t: i for i, t in enumerate(sorted(set(tokens)))}
    return vocab, {i: t for t, i in vocab.items()}

def make_pairs(tokens, window=4):
    seq = list(tokens) + [EOS]
    return [(seq[i:i + window], seq[i + window])
            for i in range(len(seq) - window)]

## 연습 6-1

다음은 김소월의 시, <엄마야 누나야>를 한 줄로 붙여 쓴 문자열이다.

*코드 6-4 김소월의 시, '엄마야 누나야'*

```python
poem = '엄마야 누나야 강변 살자 ' \
       '뜰에는 반짝이는 금모래빛 ' \
       '뒷문 밖에는 갈잎의 노래 ' \
       '엄마야 누나야 강변 살자'
```

이 문자열의 어휘 사전을 만들어 보자. 단 다음 조건에 따라 두 종류의 어휘 사전을 만든다.

각 글자(공백 문자 포함)를 토큰으로 하는 어휘 사전

각 어절을 토큰으로 하는 어휘 사전

In [ ]:
# 1) 글자 단위 어휘 사전 (공백 포함)
char_tokens = list(POEM) + [EOS]
char_vocab, char_rev = build_vocab(char_tokens)
print(f'글자 토큰 어휘 사전: {len(char_vocab)}개')
print(f'  {char_vocab}')

# 2) 어절 단위 어휘 사전
word_tokens = POEM.split() + [EOS]
word_vocab, word_rev = build_vocab(word_tokens)
print(f'\n어절 토큰 어휘 사전: {len(word_vocab)}개')
print(f'  {word_vocab}')

같은 텍스트라도 토큰 단위에 따라 어휘 사전 크기가 크게 달라진다. 글자 단위는 사전이 작지만 문장 하나를 표현하는 데 토큰이 많이 필요하고, 어절 단위는 그 반대다. 종료 특수 토큰(`<eos>`)도 사전에 포함한다.

## 연습 6-2

<엄마야 누나야>를 길이 4의 입력과 길이 1의 정답 쌍으로 재구성해 출력해 보자. [연습 문제 6-1]과 마찬가지로 각 글자가 토큰인 경우와 각 어절이 토큰인 경우 각각 출력하며, 시의 마지막은 종료 특수 토큰으로 끝나야 한다.

In [ ]:
for name, tokens in [('글자', list(POEM)), ('어절', POEM.split())]:
    pairs = make_pairs(tokens, window=4)
    print(f'=== {name} 토큰: 쌍 {len(pairs)}개 ===')
    for x, y in pairs[:3]:
        print(f'  입력 {x} -> 정답 {y!r}')
    print('  ...')
    print(f'  입력 {pairs[-1][0]} -> 정답 {pairs[-1][1]!r}   <- 마지막은 종료 토큰')
    print()

길이 4의 슬라이딩 윈도우를 한 칸씩 옮기며 (입력 4개, 정답 1개) 쌍을 만든다. 마지막 쌍의 정답이 `<eos>`가 되도록 토큰 목록 끝에 종료 토큰을 붙였다.

## 연습 6-3

[연습 문제 6-2]의 입력과 정답 쌍 중 처음과 마지막 쌍을 원-핫 벡터로 인코딩해 출력해 보자.

In [ ]:
for name, tokens, vocab in [('글자', list(POEM), char_vocab),
                            ('어절', POEM.split(), word_vocab)]:
    pairs = make_pairs(tokens, window=4)
    print(f'=== {name} 토큰 (어휘 {len(vocab)}개) ===')
    for label, (x, y) in [('처음', pairs[0]), ('마지막', pairs[-1])]:
        x_idx = torch.tensor([vocab[t] for t in x])
        y_idx = torch.tensor(vocab[y])
        x_hot = nn.functional.one_hot(x_idx, len(vocab)).float()
        y_hot = nn.functional.one_hot(y_idx, len(vocab)).float()
        print(f'  [{label}] 입력 원-핫 형태 {tuple(x_hot.shape)}, '
              f'정답 원-핫 형태 {tuple(y_hot.shape)}')
        print(f'          입력 토큰 {x} -> 인덱스 {x_idx.tolist()}')
        print(f'          정답 {y!r} -> 인덱스 {y_idx.item()}')
    print()

원-핫 인코딩은 (토큰 수, 어휘 사전 크기) 형태의 행렬이 된다. 어휘 사전이 커질수록 벡터가 길어지고 대부분이 0인 희소한 표현이 되는데, 이 비효율을 해결하는 방법이 7장의 임베딩이다.